# ViHSD Mixture of Experts experiment

This Colab entry point mounts Google Drive, installs the project dependencies, and runs the root-level training and evaluation scripts. Change the experiment parameter cell before each run; there is no need to edit or push `configs/vihsd.yaml`.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'
REPOSITORY_BRANCH = 'dense-compare'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch $REPOSITORY_BRANCH $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face and W&B authentication configured from Colab Secrets.')

## 4. Set experiment parameters

Change only the values you want to test. These override the repository YAML for this run only, and the final configuration is saved with its checkpoint.

**How keys work:** use the same path as `configs/vihsd.yaml`, replacing YAML nesting with dots. For example, `model.num_experts` changes `model: -> num_experts:`. Any omitted key keeps its YAML default.

**Example:** to compare an 8-expert, top-2 router, set `model.num_experts` to `8` and `model.top_k` to `2`; use `RUN_ID = 'experts-8-topk-2'`. Keep `top_k` less than or equal to `num_experts`. Set `EXPERIMENT_OVERRIDES = {}` to use the untouched baseline.

### Experiment overrides

An override uses the same path as a value in `configs/vihsd.yaml`, with sections separated by dots. The YAML file is never modified: omitted values keep their defaults.

For example, this Colab configuration compares an 8-expert, top-2 router against the default 4-expert, top-1 model:

```python
EXPERIMENT_OVERRIDES = {
    "training.epochs": 10,
    "training.learning_rate": 0.0001,
    "model.num_experts": 8,
    "model.top_k": 2,
}
RUN_ID = "experts-8-topk-2-lr-1e-4"
SMOKE_TEST = False
```

Set `EXPERIMENT_OVERRIDES = {}` to run the unmodified YAML defaults. Use a unique `RUN_ID` for a readable experiment folder, or set it to `None` for an automatic UTC timestamp. Keep `model.top_k` no greater than `model.num_experts`.

### What to tune first

| Priority | Settings                                                                      | Compact guidance                                                                                                                                                   |
| -------- | ----------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| 1        | `training.learning_rate`, `training.epochs`                                   | Tune learning rate first (`5e-5`, `1e-4`, `2e-4` are useful starting points), then train long enough for validation performance to plateau.                        |
| 2        | `model.model_dim`, `model.num_layers`, `dataset.max_length`                   | Main model/context capacity. Increasing them may improve accuracy, but costs GPU memory and training time. `model_dim` must be divisible by `num_attention_heads`. |
| 3        | `training.weight_decay`, `model.dropout`                                      | Regularization. Increase if training metrics improve while validation metrics worsen.                                                                              |
| 4        | `model.num_experts`, `model.top_k`, `routing.load_balance_loss_factor`        | MoE behavior. Start at `4/1/0.01`; test `8/1` then `8/2`. Keep `1 <= top_k <= num_experts`; increase balance loss if routing collapses to a few experts.           |
| 5        | `model.expert_hidden_dim`, `model.num_attention_heads`, `training.batch_size` | Secondary capacity/optimization controls. Larger batch sizes may require learning-rate retuning.                                                                   |

`seed` affects repeatability, not the expected average score; use several seeds when comparing final candidates. `max_train_samples` and `smoke_test` are for fast debugging rather than final experiments. `num_workers`, output paths, and W&B settings do not change model quality. `routing.capacity_factor` is currently not used by the code, so changing it has no effect.

Useful starting sweep: learning rate `5e-5`, `1e-4`, `2e-4`; then compare `num_experts/top_k` as `4/1`, `8/1`, and `8/2`. Higher `model_dim`, layers, length, and expert size can improve capacity but use more GPU memory/time. Increase dropout or weight decay if training improves while validation worsens.

Constraints: `model_dim` must divide evenly by `num_attention_heads`; `1 <= top_k <= num_experts`. `seed` is for repeatability; `smoke_test` and `max_train_samples` are for quick checks, not final scoring. `routing.capacity_factor` is currently unused by the code, so do not tune it.

In [ ]:
# Baseline: leave this empty to use every YAML default.
EXPERIMENT_OVERRIDES = {}

# Example experiment (uncomment this block to use it):
# EXPERIMENT_OVERRIDES = {
#     'training.epochs': 10,
#     'training.learning_rate': 0.0001,
#     'model.num_experts': 8,
#     'model.top_k': 2,
# }
RUN_ID = None  # e.g. 'experts-8-topk-2'; None creates a UTC timestamp
SMOKE_TEST = False

## 5. Train the full-parameter MoE

All embeddings, attention layers, router parameters, expert parameters, and classifier parameters are optimized. The best validation checkpoint and its resolved configuration are saved to Google Drive under a run folder. At the end of training, the notebook prints train, validation, and test loss/accuracy for the best-validation epoch and saves them as `run_metrics.json`.

In [ ]:
import json
import sys
from train import main as train_main

command = ['--config', 'configs/vihsd.yaml']
command.append('--smoke-test' if SMOKE_TEST else '--no-smoke-test')
if RUN_ID:
    command.extend(['--run-id', RUN_ID])
for key, value in EXPERIMENT_OVERRIDES.items():
    command.extend(['--set', f'{key}={json.dumps(value)}'])
print('Running: python train.py', ' '.join(command))
# Run in this notebook kernel so tqdm.auto displays live progress bars.
previous_argv = sys.argv
try:
    sys.argv = ['train.py', *command]
    train_main()
finally:
    sys.argv = previous_argv

## 6. Evaluate the newest run and save JSON predictions

This loads the latest checkpoint from Drive, evaluates the test split, records expert routing counts, and writes results into the cloned repository.

In [ ]:
!CHECKPOINT_DIR=$CHECKPOINT_DIR RESULTS_DIR=$RESULTS_DIR python evaluate.py --config configs/vihsd.yaml